# 🤖 03_voice_controller.ipynb — Therapeía Voice Controller + NLP
**Run on Raspberry Pi 4.**

## الفرق عن النسخة القديمة
النسخة القديمة كانت بتعتمد على **keyword matching** — يعني لازم تقول "start" بالظبط.

النسخة دي بتستخدم **NLP Intent Classifier** — الروبوت بيفهم **المعنى** مش الكلمة:
- `"let's go"` → `start`
- `"please do a scan"` → `scan`
- `"go into standby"` → `sleep`
- `"كم كتافلام عندك"` → `quantity`

## State Machine:
```
IDLE     → wake intent      → ACTIVE
ACTIVE   → start intent     → RUNNING
ACTIVE   → stop intent      → IDLE
RUNNING  → scan intent      → (capture + classify)
RUNNING  → stop intent      → IDLE (بعد ما يخلص)
Any      → sleep intent     → SLEEPING
Any      → inventory intent → (report)
Any      → quantity intent  → (specific medicine count)
```

## المتطلبات:
- `models/best_vision.tflite` و `models/labels.json` من **01_train_vision.ipynb**
- `models/intent_cnn.pt` و `models/nlp_meta.json` من **02_nlp_intent_classifier.ipynb**

## Cell 1 — Imports

In [6]:
import time, json, re, os
import numpy as np
import cv2
import serial
import speech_recognition as sr
import pyttsx3
import torch
import torch.nn as nn

try:
    import sounddevice as sd
    AUDIO_BACKEND = 'sounddevice'
except ImportError:
    sd = None
    AUDIO_BACKEND = 'none'

try:
    import tflite_runtime.interpreter as tflite
    print('tflite_runtime  ✅  Pi mode')
except ImportError:
    import tensorflow.lite as tflite
    print('tensorflow.lite ✅  PC mode')

print('OpenCV          :', cv2.__version__)
print('Audio backend   :', AUDIO_BACKEND)

tensorflow.lite ✅  PC mode
OpenCV          : 4.11.0
Audio backend   : none


## Cell 2 — Config

In [7]:
# ── Vision ────────────────────────────────────────────────────────────────────
TFLITE_MODEL_PATH    = 'models/best_vision.tflite'   # من 01_train_vision
LABELS_PATH          = 'models/labels.json'
IMAGE_SIZE           = (224, 224)
CONFIDENCE_THRESHOLD = 0.80
CAM_INDEX            = 0
COUNTDOWN_SEC        = 3

# ── NLP ───────────────────────────────────────────────────────────────────────
NLP_META_PATH   = 'models/nlp_meta.json'
NLP_MODEL_TYPE  = 'cnn'    # 'cnn' أو 'lstm' (cnn أسرع على Pi)
INTENT_THRESHOLD = 0.50    # minimum confidence للـ intent

# ── Serial ────────────────────────────────────────────────────────────────────
SERIAL_PORT     = '/dev/ttyUSB0'
SERIAL_BAUDRATE = 9600
SERIAL_TIMEOUT  = 2

SERIAL_COMMANDS = {
    'ketofan'   : 'A',
    'brufen'    : 'B',
    'cataflam'  : 'C',
    'ambezim'   : 'D',
    'trimed_flu': 'S',
    'sleep'     : 'D',
    'stop'      : 'D',
    'start'     : 'S', 
}

# ── Inventory ─────────────────────────────────────────────────────────────────
INVENTORY_PATH = 'inventory.json'
ROBOT_NAME     = 'Therapeía'

# ── Beep ──────────────────────────────────────────────────────────────────────
BEEP_SAMPLE_RATE = 44100
BEEP_WAKE   = dict(freq=880,  duration=0.15, volume=0.4)
BEEP_SCAN   = dict(freq=660,  duration=0.12, volume=0.3)
BEEP_RESULT = dict(freq=1047, duration=0.20, volume=0.4)
BEEP_ERROR  = dict(freq=330,  duration=0.25, volume=0.4)
BEEP_SLEEP  = dict(freq=440,  duration=0.30, volume=0.3)

print('Config loaded ✅')

Config loaded ✅


## Cell 3 — NLP Intent Classifier (من 02)

In [8]:
# ── Model Architecture (نفس 02_nlp) ──────────────────────────────────────────

class TextCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=64,
                 num_classes=8, num_filters=64,
                 kernel_sizes=(2, 3, 4)):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.convs     = nn.ModuleList([
            nn.Conv1d(embed_dim, num_filters, k) for k in kernel_sizes
        ])
        self.dropout   = nn.Dropout(0.4)
        self.fc        = nn.Linear(num_filters * len(kernel_sizes), num_classes)

    def forward(self, x):
        x = self.embedding(x).permute(0, 2, 1)
        x = [torch.relu(c(x)).max(-1)[0] for c in self.convs]
        x = torch.cat(x, dim=1)
        return self.fc(self.dropout(x))


class BiLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=64,
                 hidden=128, num_layers=2,
                 num_classes=8):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm      = nn.LSTM(embed_dim, hidden,
                                 num_layers=num_layers,
                                 batch_first=True,
                                 bidirectional=True,
                                 dropout=0.3)
        self.dropout   = nn.Dropout(0.4)
        self.fc        = nn.Linear(hidden * 2, num_classes)

    def forward(self, x):
        x, _ = self.lstm(self.embedding(x))
        return self.fc(self.dropout(x[:, -1, :]))


# ── Load NLP model ────────────────────────────────────────────────────────────
with open(NLP_META_PATH) as f:
    nlp_meta = json.load(f)

_word2idx    = nlp_meta['word2idx']
_intent_names = nlp_meta['intent_names']
_max_len     = nlp_meta['max_len']
_num_classes = nlp_meta['num_classes']

if NLP_MODEL_TYPE == 'cnn':
    nlp_model = TextCNN(len(_word2idx), num_classes=_num_classes)
    nlp_model.load_state_dict(torch.load('models/intent_cnn.pt', map_location='cpu'))
else:
    nlp_model = BiLSTM(len(_word2idx), num_classes=_num_classes)
    nlp_model.load_state_dict(torch.load('models/intent_lstm.pt', map_location='cpu'))

nlp_model.eval()
print(f'NLP model ({NLP_MODEL_TYPE.upper()}) loaded ✅')
print(f'Intents   : {_intent_names}')


def predict_intent(text: str, threshold=INTENT_THRESHOLD):
    """
    يستبدل keyword matching بالكامل.
    Returns: (intent_str, confidence_float)
    """
    text = re.sub(r"[^a-z0-9\s'\u0600-\u06FF]", ' ', text.lower())
    text = re.sub(r'\s+', ' ', text).strip()

    ids = [_word2idx.get(w, 1) for w in text.split()][:_max_len]
    ids += [0] * (_max_len - len(ids))
    x   = torch.tensor([ids], dtype=torch.long)

    with torch.no_grad():
        probs = torch.softmax(nlp_model(x), dim=1)[0]

    top_conf = probs.max().item()
    top_idx  = probs.argmax().item()

    if top_conf < threshold:
        return 'unknown', top_conf

    return _intent_names[top_idx], top_conf

NLP model (CNN) loaded ✅
Intents   : ['inventory', 'medicine_name', 'quantity', 'scan', 'sleep', 'start', 'stop', 'wake']


## Cell 4 — Beep Engine

In [9]:
def _make_tone(freq, duration, volume):
    t    = np.linspace(0, duration, int(BEEP_SAMPLE_RATE * duration), endpoint=False)
    wave = np.sin(2 * np.pi * freq * t).astype(np.float32)
    fade = int(BEEP_SAMPLE_RATE * 0.01)
    wave[:fade]  *= np.linspace(0, 1, fade)
    wave[-fade:] *= np.linspace(1, 0, fade)
    return wave * volume

def beep(preset, repeat=1, gap=0.08):
    if sd is None:
        return
    tone = _make_tone(**preset)
    for _ in range(repeat):
        sd.play(tone, BEEP_SAMPLE_RATE)
        sd.wait()
        if repeat > 1:
            time.sleep(gap)

print('Beep engine ✅')

Beep engine ✅


## Cell 5 — TTS Engine

In [10]:
_tts = pyttsx3.init()
_tts.setProperty('rate', 150)
_tts.setProperty('volume', 1.0)

_voices = _tts.getProperty('voices')
_female_kw = ['female','zira','hazel','susan','helen','victoria','samantha','karen']
_female_id = next(
    (v.id for v in _voices if any(kw in v.name.lower() for kw in _female_kw)),
    _voices[1].id if len(_voices) > 1 else _voices[0].id,
)
_tts.setProperty('voice', _female_id)

def speak(text: str):
    print(f'[🔊 TTS] {text}')
    _tts.say(text)
    _tts.runAndWait()

print('TTS engine ✅')

TTS engine ✅


## Cell 6 — Speech Recognition

In [11]:
_rec = sr.Recognizer()
_rec.energy_threshold         = 300
_rec.dynamic_energy_threshold = True
_rec.pause_threshold          = 0.7

def listen(timeout=7, phrase_limit=8) -> str:
    """يسمع ويرجع النص. لو فشل يرجع ''"""
    with sr.Microphone() as source:
        _rec.adjust_for_ambient_noise(source, duration=0.3)
        try:
            audio = _rec.listen(source, timeout=timeout, phrase_time_limit=phrase_limit)
            text  = _rec.recognize_google(audio).lower()
            print(f'[🎤 Heard] "{text}"')
            return text
        except sr.WaitTimeoutError:
            return ''
        except sr.UnknownValueError:
            return ''
        except Exception as e:
            print(f'[🎤 Error] {e}')
            return ''

print('Speech recognizer ✅')

Speech recognizer ✅


## Cell 7 — Vision Model

In [12]:
interpreter = tflite.Interpreter(model_path=TFLITE_MODEL_PATH, num_threads=4)
interpreter.allocate_tensors()
_inp = interpreter.get_input_details()
_out = interpreter.get_output_details()

with open(LABELS_PATH) as f:
    labels = {int(k): v for k, v in json.load(f).items()}

# قراءة نوع preprocess من meta لو موجود
try:
    with open('models/best_vision_meta.json') as f:
        vision_meta     = json.load(f)
    PREPROCESS_TYPE = vision_meta.get('preprocess_type', 'mobilenet')
    print(f'Vision model : {vision_meta["best_model"]}  '
          f'(acc={vision_meta["best_acc"]:.4f})')
except FileNotFoundError:
    PREPROCESS_TYPE = 'mobilenet'
    print('Vision model loaded (no meta found, assuming mobilenet preprocessing)')

print(f'Labels : {labels}')

Vision model : MobileNetV2  (acc=1.0000)
Labels : {0: 'ambezim', 1: 'brufen', 2: 'cataflam', 3: 'ketofan', 4: 'trimed_flu'}


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


## Cell 8 — Serial & Inventory

In [13]:
# ── Serial ────────────────────────────────────────────────────────────────────
try:
    ser = serial.Serial(SERIAL_PORT, SERIAL_BAUDRATE, timeout=SERIAL_TIMEOUT)
    time.sleep(2)
    print(f'Serial ✅  {SERIAL_PORT}')
except serial.SerialException as e:
    ser = None
    print(f'Serial ⚠️  {e}  — running without Arduino')

def send_serial(label: str):
    # بنحول الكلمة لـ lowercase وبنبدل المسافة بـ underscore عشان تطابق القاموس
    clean_label = label.lower().strip().replace(" ", "_")
    cmd = SERIAL_COMMANDS.get(clean_label, '?')
    
    print(f'[📡 Serial] → {repr(cmd)}  ({label})')
    if ser and ser.is_open:
        # بنبعت الحرف encode مباشرة "خام" بدون \n
        ser.write(cmd.encode())
        time.sleep(0.1)

def send_sleep_cmd():
    if ser and ser.is_open:
        # في كود الأردوينو بتاعك حرف D هو اللي بيوقف المحركات
        ser.write(b'D')
    print('[📡 Serial] → Action: Stop/Sleep (D)')

def send_home_cmd():
    if ser and ser.is_open:
        # في كود الأردوينو حرف S بيبدأ الـ Line Tracking للعودة للمحطة
        ser.write(b'S')
    print('[📡 Serial] → Action: Home/LineTracking (S)')

# ── Inventory ─────────────────────────────────────────────────────────────────
_DEFAULT_INV = {k: 10 for k in labels.values()}

if not os.path.exists(INVENTORY_PATH):
    with open(INVENTORY_PATH, 'w') as f:
        json.dump(_DEFAULT_INV, f, indent=2)

def load_inventory():
    with open(INVENTORY_PATH) as f:
        return json.load(f)

def save_inventory(inv):
    with open(INVENTORY_PATH, 'w') as f:
        json.dump(inv, f, indent=2)

def decrement_inventory(medicine: str) -> int:
    inv = load_inventory()
    current = inv.get(medicine, 0)
    if current > 0:
        inv[medicine] = current - 1
        save_inventory(inv)
        return current - 1
    return 0

def speak_inventory():
    inv = load_inventory()
    speak('Current inventory report:')
    for med, qty in inv.items():
        speak(f'{med}: {qty} units')

def speak_quantity(medicine: str):
    inv = load_inventory()
    # بحث fuzzy عن اسم الدواء في النص
    for med in inv:
        if med.replace('_', ' ') in medicine or med in medicine:
            qty = inv[med]
            speak(f'We have {qty} units of {med}.')
            return
    speak('I could not identify which medicine you are asking about.')

print('Serial & Inventory ✅')

Serial ⚠️  [Errno 2] could not open port /dev/ttyUSB0: [Errno 2] No such file or directory: '/dev/ttyUSB0'  — running without Arduino
Serial & Inventory ✅


## Cell 9 — Camera & Classify

In [14]:
from IPython.display import display as ipy_display, clear_output
import PIL.Image

def camera_preview_and_capture() -> np.ndarray:
    """Countdown + capture. Returns BGR frame."""
    cap = cv2.VideoCapture(CAM_INDEX)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH,  640)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

    start, captured_frame, last_count = time.time(), None, -1

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        elapsed   = time.time() - start
        remaining = COUNTDOWN_SEC - int(elapsed)
        disp      = frame.copy()
        h, w      = disp.shape[:2]

        if remaining > 0:
            if remaining != last_count:
                beep(BEEP_SCAN)
                last_count = remaining
            cv2.putText(disp, str(remaining), (w//2-40, h//2+20),
                        cv2.FONT_HERSHEY_DUPLEX, 4, (0,255,0), 6)
        else:
            captured_frame = frame.copy()
            cv2.putText(disp, 'CAPTURED!', (50, h//2),
                        cv2.FONT_HERSHEY_DUPLEX, 2, (0,0,255), 4)

        # Show in Jupyter
        rgb  = cv2.cvtColor(disp, cv2.COLOR_BGR2RGB)
        img  = PIL.Image.fromarray(rgb).resize((480, 360))
        clear_output(wait=True)
        ipy_display(img)

        if captured_frame is not None:
            time.sleep(0.5)
            break

    cap.release()
    return captured_frame


def preprocess_frame(frame: np.ndarray) -> np.ndarray:
    h, w = IMAGE_SIZE
    img  = cv2.resize(frame, (w, h))
    img  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32)
    if PREPROCESS_TYPE == 'efficientnet':
        # EfficientNet: scale to [-1, 1]
        img = (img / 127.5) - 1.0
    else:
        # MobileNetV2: scale to [0, 1]
        img = img / 255.0
    return np.expand_dims(img, axis=0)


def classify(frame: np.ndarray):
    """Returns (label, confidence) or (None, 0) if below threshold."""
    img = preprocess_frame(frame)
    interpreter.set_tensor(_inp[0]['index'], img)
    interpreter.invoke()
    probs = interpreter.get_tensor(_out[0]['index'])[0]

    top_idx  = int(np.argmax(probs))
    top_conf = float(probs[top_idx])
    label    = labels[top_idx]

    print('\n── Vision Result ─────────────────────────────────')
    for idx in sorted(labels):
        bar = '█' * int(probs[idx] * 20)
        mrk = ' ◄' if idx == top_idx else ''
        print(f'  {labels[idx]:<14} {probs[idx]:.2%}  {bar}{mrk}')
    print(f'  Confidence : {top_conf:.2%}')

    return (label, top_conf) if top_conf >= CONFIDENCE_THRESHOLD else (None, top_conf)

print('Camera & Classify ✅')

Camera & Classify ✅


## Cell 10 — 🚀 Main Loop

In [15]:
# ══════════════════════════════════════════════════════════════════════════════
#  STATE MACHINE
#  States: IDLE | ACTIVE | RUNNING | SLEEPING
# ══════════════════════════════════════════════════════════════════════════════

STATE = 'IDLE'

print('=' * 56)
print(f'  🤖  {ROBOT_NAME}  —  Medical Assistant (NLP Edition)')
print('=' * 56)
print('  الـ NLP بيفهم المعنى — مش محتاج تقول كلمة بعينها!')
print('  أمثلة:')
print('    wake    : therapeia / hey robot / يا ثيرابيا')
print('    start   : start / let\'s go / ابدأ')
print('    scan    : scan / take photo / افحص')
print('    stop    : stop / enough / وقف')
print('    sleep   : sleep / go home / نام')
print('    inventory: report / ما عندك / show inventory')
print('    quantity : how many brufen / كم كتافلام')
print('=' * 56)
print('  Press Jupyter Stop button to exit')
print('=' * 56)

speak(f'{ROBOT_NAME} is online. Say my name to activate.')
beep(BEEP_WAKE, repeat=2)

while True:
    # ── Prompt based on state ─────────────────────────────────────────────────
    if STATE == 'IDLE':
        print(f'\n[STATE: IDLE] Listening for wake word...')
        text = listen(timeout=10)
    elif STATE == 'ACTIVE':
        print(f'\n[STATE: ACTIVE] What should I do?')
        text = listen(timeout=10)
    elif STATE == 'RUNNING':
        print(f'\n[STATE: RUNNING] Say "scan", "stop", or any command...')
        text = listen(timeout=8)
    elif STATE == 'SLEEPING':
        print(f'\n[STATE: SLEEPING] Say my name to wake up...')
        text = listen(timeout=15)
    else:
        text = listen()

    if not text:
        continue

    # ── Classify intent with NLP ──────────────────────────────────────────────
    intent, confidence = predict_intent(text)
    print(f'[🧠 NLP] "{text}" → intent={intent}  conf={confidence:.2%}')

    # ══════════════════════════
    #  PRIORITY 1: SLEEP (interrupts anything)
    # ══════════════════════════
    if intent == 'sleep':
        beep(BEEP_SLEEP)
        speak(f'Returning to home position and entering sleep mode.')
        send_sleep_cmd()
        STATE = 'SLEEPING'
        continue

    # ══════════════════════════
    #  PRIORITY 2: INVENTORY
    # ══════════════════════════
    if intent == 'inventory':
        speak_inventory()
        continue

    # ══════════════════════════
    #  PRIORITY 3: QUANTITY
    # ══════════════════════════
    if intent == 'quantity':
        speak_quantity(text)
        continue

    # ══════════════════════════
    #  PRIORITY 4: MEDICINE NAME (dispatch directly)
    # ══════════════════════════
    if intent == 'medicine_name' and STATE == 'RUNNING':
        # استخرج اسم الدواء من النص مباشرةً
        detected_med = None
        for med in labels.values():
            if med.replace('_', ' ') in text or med in text:
                detected_med = med
                break
        if detected_med:
            remaining = decrement_inventory(detected_med)
            send_serial(detected_med)
            beep(BEEP_RESULT)
            speak(f'Dispensing {detected_med}. {remaining} units remaining.')
        continue

    # ══════════════════════════
    #  STATE: SLEEPING
    # ══════════════════════════
    if STATE == 'SLEEPING':
        if intent == 'wake':
            beep(BEEP_WAKE)
            speak(f'{ROBOT_NAME} is awake. What should I do?')
            STATE = 'ACTIVE'
        continue

    # ══════════════════════════
    #  STATE: IDLE
    # ══════════════════════════
    if STATE == 'IDLE':
        if intent == 'wake':
            beep(BEEP_WAKE)
            speak(f'Hello! {ROBOT_NAME} is active. Say start to begin.')
            STATE = 'ACTIVE'
        continue

    # ══════════════════════════
    #  STATE: ACTIVE
    # ══════════════════════════
    if STATE == 'ACTIVE':
        if intent == 'start':
            beep(BEEP_WAKE, repeat=2)
            speak('System started. I will classify medicines now.')
            STATE = 'RUNNING'
        elif intent == 'stop':
            speak(f'{ROBOT_NAME} is now idle.')
            STATE = 'IDLE'
        elif intent == 'wake':
            speak("I'm already active! Say start to begin scanning.")
        elif intent == 'unknown':
            speak("I did not understand. Please say start or stop.")
        continue

    # ══════════════════════════
    #  STATE: RUNNING
    # ══════════════════════════
    if STATE == 'RUNNING':
        if intent == 'stop':
            speak(f'Stopping after current operation.')
            STATE = 'IDLE'

        elif intent in ('scan', 'start'):
            beep(BEEP_SCAN, repeat=2)
            speak('Capturing image now.')
            frame = camera_preview_and_capture()

            if frame is None:
                beep(BEEP_ERROR)
                speak('Camera failed. Please try again.')
                continue

            label, conf = classify(frame)

            if label is None:
                beep(BEEP_ERROR)
                speak(f'Confidence too low: {conf:.0%}. Please try again.')
            else:
                remaining = decrement_inventory(label)
                send_serial(label)
                beep(BEEP_RESULT)
                speak(f'Medicine detected: {label}. Confidence {conf:.0%}. '
                      f'{remaining} units remaining.')

        elif intent == 'wake':
            speak("I'm already running! Say scan to classify a medicine.")

        elif intent == 'unknown':
            speak("I did not understand. Say scan or stop.")

  🤖  Therapeía  —  Medical Assistant (NLP Edition)
  الـ NLP بيفهم المعنى — مش محتاج تقول كلمة بعينها!
  أمثلة:
    wake    : therapeia / hey robot / يا ثيرابيا
    start   : start / let's go / ابدأ
    scan    : scan / take photo / افحص
    stop    : stop / enough / وقف
    sleep   : sleep / go home / نام
    inventory: report / ما عندك / show inventory
    quantity : how many brufen / كم كتافلام
  Press Jupyter Stop button to exit
[🔊 TTS] Therapeía is online. Say my name to activate.

[STATE: IDLE] Listening for wake word...


KeyboardInterrupt: 